# DINOv3 Object Detection Training and Benchmark on Garbage COCO

This notebook uses the official `facebookresearch/dinov3` repository. DINOv3 itself is a visual foundation backbone, so the detector here is `DINOv3 ConvNeXt-Tiny + FPN + Faster R-CNN`. It trains on the Roboflow COCO dataset and benchmarks on the test split.


Kaggle notes: enable GPU and Internet. Add the exported Roboflow `COCO JSON` dataset as an input dataset. If the official DINOv3 weights cannot be downloaded, upload the DINOv3 ConvNeXt-Tiny `.pth` weight file as another Kaggle dataset and set `DINOV3_BACKBONE_WEIGHTS` in the config cell.


In [1]:
import os
import sys
import json
import time
import shutil
import subprocess
from collections import OrderedDict, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision.transforms import functional as F

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def run(cmd, cwd=None, check=True):
    printable = " ".join(str(x) for x in cmd) if isinstance(cmd, (list, tuple)) else cmd
    print(f"\n$ {printable}")
    return subprocess.run(cmd, cwd=cwd, check=check)


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# DINOv3 is a visual foundation backbone, not a ready-to-fine-tune garbage detector.
# This notebook trains an object detector using:
#   DINOv3 ConvNeXt-Tiny backbone + Feature Pyramid Network + Faster R-CNN detection head.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "DINOv3 ConvNeXt-Tiny + Faster R-CNN"
MODEL_SLUG = "dinov3_convnext_tiny_fasterrcnn"

DATASET_SRC = None  # Optional: "/kaggle/input/.../GARBAGE CLASSIFICATION 3.v2-gc1.coco"
REPO_DIR = WORK_DIR / "facebookresearch_dinov3"
PREPARED_COCO_DIR = WORK_DIR / "garbage_coco_dinov3_layout"
OUTPUT_DIR = WORK_DIR / "outputs_dinov3_fasterrcnn"
BENCHMARK_DIR = WORK_DIR / "benchmark_results"

RUN_TRAIN = True
RESUME_FROM_LAST = True

# If None, torch.hub uses the official default LVD-1689M DINOv3 weights.
# You can also point this to a Hugging Face model.safetensors file; the notebook converts ConvNeXt-Tiny keys.
# If Kaggle cannot download the weights, upload the .pth or .safetensors file as a Kaggle dataset and set this path manually.
DINOV3_BACKBONE_WEIGHTS = '/kaggle/input/models/minhkhoathi/dinov3-convnext-tiny-pretrain-lvd1689m/pytorch/default/1/model.safetensors'
USE_DINOV3_PRETRAINED = True
ALLOW_RANDOM_BACKBONE_FALLBACK = False

EPOCHS = 10
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
NUM_WORKERS = 2

IMAGE_MIN_SIZE = 416
IMAGE_MAX_SIZE = 672

LR_HEAD = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

# Good default for a T4: freeze most of the DINOv3 backbone and train the detector head.
# Set to False if you want full fine-tuning and have enough GPU memory/time.
FREEZE_BACKBONE = True
UNFREEZE_LAST_STAGE = False

CONF_THRESHOLD = 0.50
IOU_THRESHOLD = 0.50

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Output:", OUTPUT_DIR)


Model: DINOv3 ConvNeXt-Tiny + Faster R-CNN
Device: cuda
Output: /kaggle/working/outputs_dinov3_fasterrcnn


In [3]:
run([sys.executable, "-m", "pip", "install", "-q", "pycocotools", "scipy", "pandas", "matplotlib", "tqdm", "omegaconf", "torchmetrics", "safetensors"])

if not REPO_DIR.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/facebookresearch/dinov3.git", str(REPO_DIR)])
else:
    print("Repo already exists:", REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))



$ /usr/bin/python3 -m pip install -q pycocotools scipy pandas matplotlib tqdm omegaconf torchmetrics safetensors

$ git clone --depth 1 https://github.com/facebookresearch/dinov3.git /kaggle/working/facebookresearch_dinov3


Cloning into '/kaggle/working/facebookresearch_dinov3'...


In [4]:
# Prepare Roboflow COCO dataset for torchvision training.
# Source:
#   root/train/_annotations.coco.json
#   root/valid/_annotations.coco.json
#   root/test/_annotations.coco.json
# Target:
#   PREPARED_COCO_DIR/train
#   PREPARED_COCO_DIR/valid
#   PREPARED_COCO_DIR/test
#   PREPARED_COCO_DIR/annotations/instances_*.json
#
# Labels are remapped to 1..N because torchvision detectors reserve label 0 for background.

def find_roboflow_coco_root():
    if DATASET_SRC:
        root = Path(DATASET_SRC)
        if all((root / s / "_annotations.coco.json").exists() for s in ["train", "valid", "test"]):
            return root
        raise FileNotFoundError(f"DATASET_SRC does not look like Roboflow COCO: {root}")

    candidates = []
    if INPUT_DIR.exists():
        for ann in INPUT_DIR.rglob("_annotations.coco.json"):
            if ann.parent.name in {"train", "valid", "test"}:
                candidates.append(ann.parent.parent)

    local_candidate = Path(r"D:\Garbage object detection\GARBAGE CLASSIFICATION 3.v2-gc1.coco")
    candidates.extend([Path.cwd(), WORK_DIR, local_candidate])

    seen = set()
    for root in candidates:
        root = Path(root)
        key = str(root.resolve()) if root.exists() else str(root)
        if key in seen:
            continue
        seen.add(key)
        if all((root / s / "_annotations.coco.json").exists() for s in ["train", "valid", "test"]):
            return root

    raise FileNotFoundError(
        "Cannot find Roboflow COCO dataset. On Kaggle, add the COCO JSON dataset under /kaggle/input, "
        "or set DATASET_SRC manually."
    )


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f)


def link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def prepare_coco_layout(src_root, dst_root):
    split_to_ann = {
        "train": "instances_train.json",
        "valid": "instances_valid.json",
        "test": "instances_test.json",
    }
    raw = {split: load_json(src_root / split / "_annotations.coco.json") for split in split_to_ann}

    active_cat_ids = sorted({
        ann["category_id"]
        for data in raw.values()
        for ann in data.get("annotations", [])
    })
    cat_meta = {}
    for data in raw.values():
        for cat in data.get("categories", []):
            cat_meta[cat["id"]] = cat

    old_to_new = {old_id: new_id + 1 for new_id, old_id in enumerate(active_cat_ids)}
    categories = []
    for old_id in active_cat_ids:
        cat = dict(cat_meta[old_id])
        cat["id"] = old_to_new[old_id]
        categories.append(cat)

    dst_root = Path(dst_root)
    ann_dir = dst_root / "annotations"
    ann_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for split, data in raw.items():
        img_dst_dir = dst_root / split
        img_dst_dir.mkdir(parents=True, exist_ok=True)
        images = data.get("images", [])
        image_ids = {img["id"] for img in images}
        annotations = []
        ann_id = 1

        for img in images:
            src_img = src_root / split / img["file_name"]
            if not src_img.exists():
                matches = list((src_root / split).rglob(img["file_name"]))
                if matches:
                    src_img = matches[0]
            if not src_img.exists():
                raise FileNotFoundError(f"Missing image: {src_root / split / img['file_name']}")
            link_or_copy(src_img, img_dst_dir / img["file_name"])

        for ann in data.get("annotations", []):
            old_cat = ann.get("category_id")
            if old_cat not in old_to_new or ann.get("image_id") not in image_ids:
                continue
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            new_ann = dict(ann)
            new_ann["id"] = ann_id
            new_ann["category_id"] = old_to_new[old_cat]
            new_ann["bbox"] = [float(x), float(y), float(w), float(h)]
            new_ann["area"] = float(ann.get("area", w * h))
            new_ann["iscrowd"] = int(ann.get("iscrowd", 0))
            new_ann.setdefault("segmentation", [])
            annotations.append(new_ann)
            ann_id += 1

        out = {
            "info": data.get("info", {}),
            "licenses": data.get("licenses", []),
            "images": images,
            "annotations": annotations,
            "categories": categories,
        }
        ann_path = ann_dir / split_to_ann[split]
        save_json(out, ann_path)
        rows.append({
            "split": split,
            "images": len(images),
            "annotations": len(annotations),
            "annotation_file": str(ann_path),
        })

    class_names = [c["name"] for c in categories]
    save_json({"class_names": class_names, "old_to_new": old_to_new}, dst_root / "class_mapping.json")
    return class_names, pd.DataFrame(rows)


DATASET_ROOT = find_roboflow_coco_root()
CLASS_NAMES, dataset_summary = prepare_coco_layout(DATASET_ROOT, PREPARED_COCO_DIR)
NUM_CLASSES = len(CLASS_NAMES)

print("Dataset source:", DATASET_ROOT)
print("Prepared COCO dir:", PREPARED_COCO_DIR)
print("Classes:", CLASS_NAMES)
display(dataset_summary)


Dataset source: /kaggle/input/datasets/minhkhoathi/garbage-object-detection-coco-dataset
Prepared COCO dir: /kaggle/working/garbage_coco_dinov3_layout
Classes: ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']


,split,images,annotations,annotation_file
0,train,7324,51601,/kaggle/working/garbage_coco_dinov3_layout/ann...
1,valid,2098,18910,/kaggle/working/garbage_coco_dinov3_layout/ann...
2,test,1042,3562,/kaggle/working/garbage_coco_dinov3_layout/ann...


In [5]:
from pycocotools.coco import COCO
from torchvision.ops import FeaturePyramidNetwork
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign


class CocoDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split):
        self.root_dir = Path(root_dir)
        self.split = split
        self.image_dir = self.root_dir / split
        self.ann_path = self.root_dir / "annotations" / f"instances_{split}.json"
        self.coco = COCO(str(self.ann_path))
        self.image_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(image_id)[0]
        img_path = self.image_dir / img_info["file_name"]
        image = Image.open(img_path).convert("RGB")
        image = F.to_tensor(image)

        ann_ids = self.coco.getAnnIds(imgIds=[image_id], iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)
        boxes = []
        labels = []
        areas = []
        iscrowd = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(int(ann["category_id"]))
            areas.append(float(ann.get("area", w * h)))
            iscrowd.append(int(ann.get("iscrowd", 0)))

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": torch.tensor(areas, dtype=torch.float32) if areas else torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.tensor(iscrowd, dtype=torch.int64) if iscrowd else torch.zeros((0,), dtype=torch.int64),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


def convert_hf_dinov3_convnext_tiny_state_dict(state_dict):
    """Convert Hugging Face DINOv3 ConvNeXt-Tiny safetensors keys to Meta repo keys."""
    converted = {}
    for key, value in state_dict.items():
        new_key = key
        if new_key.startswith("layer_norm."):
            new_key = new_key.replace("layer_norm.", "norm.", 1)

        for stage in range(4):
            prefix = f"stages.{stage}.downsample_layers."
            if new_key.startswith(prefix):
                rest = new_key[len(prefix):]
                new_key = f"downsample_layers.{stage}.{rest}"
                break

        new_key = new_key.replace(".layers.", ".")
        new_key = new_key.replace(".depthwise_conv.", ".dwconv.")
        new_key = new_key.replace(".layer_norm.", ".norm.")
        new_key = new_key.replace(".pointwise_conv1.", ".pwconv1.")
        new_key = new_key.replace(".pointwise_conv2.", ".pwconv2.")
        converted[new_key] = value
    return converted


def load_dinov3_hf_safetensors_into_meta_convnext(model, safetensors_path):
    from safetensors.torch import load_file

    raw_state = load_file(str(safetensors_path), device="cpu")
    converted_state = convert_hf_dinov3_convnext_tiny_state_dict(raw_state)
    model_state = model.state_dict()
    filtered_state = {
        k: v for k, v in converted_state.items()
        if k in model_state and tuple(v.shape) == tuple(model_state[k].shape)
    }
    missing, unexpected = model.load_state_dict(filtered_state, strict=False)
    print(f"Loaded safetensors tensors: {len(filtered_state)} / {len(model_state)}")
    print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")
    if len(filtered_state) < 0.9 * len(model_state):
        raise RuntimeError(
            "Too few safetensors keys matched the DINOv3 ConvNeXt-Tiny backbone. "
            "Please verify that DINOV3_BACKBONE_WEIGHTS points to a DINOv3 ConvNeXt-Tiny model.safetensors file."
        )



class DINOv3ConvNeXtTinyFPN(nn.Module):
    def __init__(self, repo_dir, pretrained=True, weights=None, freeze_backbone=True, unfreeze_last_stage=False):
        super().__init__()
        weights_path = Path(weights) if weights is not None else None
        if weights_path is not None and weights_path.suffix.lower() == ".safetensors":
            print("Loading DINOv3 ConvNeXt-Tiny from Hugging Face safetensors:", weights_path)
            self.body = torch.hub.load(str(repo_dir), "dinov3_convnext_tiny", source="local", pretrained=False)
            load_dinov3_hf_safetensors_into_meta_convnext(self.body, weights_path)
        else:
            load_kwargs = dict(source="local", pretrained=pretrained)
            if weights_path is not None:
                load_kwargs["weights"] = str(weights_path)
            try:
                self.body = torch.hub.load(str(repo_dir), "dinov3_convnext_tiny", **load_kwargs)
            except Exception as exc:
                if pretrained and ALLOW_RANDOM_BACKBONE_FALLBACK:
                    print("Could not load pretrained DINOv3 weights. Falling back to random DINOv3 ConvNeXt-Tiny.")
                    print(repr(exc))
                    self.body = torch.hub.load(str(repo_dir), "dinov3_convnext_tiny", source="local", pretrained=False)
                else:
                    raise RuntimeError(
                        "Could not load DINOv3 pretrained weights. If Kaggle cannot download them, upload the "
                        ".pth or .safetensors weight file as a Kaggle dataset and set DINOV3_BACKBONE_WEIGHTS to that path."
                    ) from exc

        if freeze_backbone:
            for p in self.body.parameters():
                p.requires_grad = False
            if unfreeze_last_stage and hasattr(self.body, "stages"):
                for p in self.body.stages[-1].parameters():
                    p.requires_grad = True
                if hasattr(self.body, "downsample_layers"):
                    for p in self.body.downsample_layers[-1].parameters():
                        p.requires_grad = True

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[96, 192, 384, 768],
            out_channels=256,
            extra_blocks=LastLevelMaxPool(),
        )
        self.out_channels = 256

    def forward(self, x):
        features = self.body.get_intermediate_layers(
            x,
            n=[0, 1, 2, 3],
            reshape=True,
            return_class_token=False,
            norm=False,
        )
        features = OrderedDict((str(i), feat) for i, feat in enumerate(features))
        return self.fpn(features)


def build_detector():
    backbone = DINOv3ConvNeXtTinyFPN(
        REPO_DIR,
        pretrained=USE_DINOV3_PRETRAINED,
        weights=DINOV3_BACKBONE_WEIGHTS,
        freeze_backbone=FREEZE_BACKBONE,
        unfreeze_last_stage=UNFREEZE_LAST_STAGE,
    )
    anchor_generator = AnchorGenerator(
        sizes=((16,), (32,), (64,), (128,), (256,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5,
    )
    roi_pooler = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3", "pool"],
        output_size=7,
        sampling_ratio=2,
    )
    model = FasterRCNN(
        backbone,
        num_classes=NUM_CLASSES + 1,
        min_size=IMAGE_MIN_SIZE,
        max_size=IMAGE_MAX_SIZE,
        image_mean=[0.485, 0.456, 0.406],
        image_std=[0.229, 0.224, 0.225],
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
    )
    return model


train_dataset = CocoDetectionDataset(PREPARED_COCO_DIR, "train")
valid_dataset = CocoDetectionDataset(PREPARED_COCO_DIR, "valid")
test_dataset = CocoDetectionDataset(PREPARED_COCO_DIR, "test")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == "cuda"),
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == "cuda"),
)

model = build_detector().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: trainable={trainable/1e6:.2f}M / total={total/1e6:.2f}M")


loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading DINOv3 ConvNeXt-Tiny from Hugging Face safetensors: /kaggle/input/models/minhkhoathi/dinov3-convnext-tiny-pretrain-lvd1689m/pytorch/default/1/model.safetensors
Loaded safetensors tensors: 180 / 182
Missing keys: 2 | Unexpected keys: 0
Parameters: trainable=17.26M / total=45.08M


In [6]:
from tqdm.auto import tqdm
from pycocotools.cocoeval import COCOeval


def move_targets_to_device(targets, device):
    return [{k: v.to(device) for k, v in t.items()} for t in targets]


def xyxy_to_xywh(box):
    x1, y1, x2, y2 = [float(x) for x in box]
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]


def xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]


def box_iou_xyxy(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, a[:, 2] - a[:, 0]) * np.maximum(0.0, a[:, 3] - a[:, 1])
    area_b = np.maximum(0.0, b[:, 2] - b[:, 0]) * np.maximum(0.0, b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.maximum(union, 1e-9)


@torch.no_grad()
def collect_predictions(model, data_loader, score_min=0.001):
    model.eval()
    detections = []
    image_count = 0
    infer_time = 0.0

    for images, targets in tqdm(data_loader, desc="Inference", leave=False):
        images = [img.to(DEVICE) for img in images]
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        infer_time += time.perf_counter() - t0
        image_count += len(images)

        for target, output in zip(targets, outputs):
            image_id = int(target["image_id"].item())
            boxes = output["boxes"].detach().cpu().numpy()
            scores = output["scores"].detach().cpu().numpy()
            labels = output["labels"].detach().cpu().numpy()
            for box, score, label in zip(boxes, scores, labels):
                if float(score) < score_min:
                    continue
                detections.append({
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": xyxy_to_xywh(box),
                    "score": float(score),
                })
    return detections, infer_time, image_count


def compute_point_metrics(coco_gt, detections, coco_eval, conf_threshold=0.5, iou_threshold=0.5):
    img_ids = sorted(coco_gt.getImgIds())
    cat_ids = sorted(coco_gt.getCatIds())
    cats = {c["id"]: c["name"] for c in coco_gt.loadCats(cat_ids)}

    gt_by_img_cat = defaultdict(list)
    for ann in coco_gt.loadAnns(coco_gt.getAnnIds()):
        if int(ann.get("iscrowd", 0)) == 1:
            continue
        gt_by_img_cat[(ann["image_id"], ann["category_id"])].append(xywh_to_xyxy(ann["bbox"]))

    pred_by_img_cat = defaultdict(list)
    for det in detections:
        if det["score"] >= conf_threshold:
            pred_by_img_cat[(det["image_id"], det["category_id"])].append(det)

    ap50_by_cat = {}
    ap5095_by_cat = {}
    if coco_eval is not None and getattr(coco_eval, "eval", None):
        precision_array = coco_eval.eval.get("precision")
        if precision_array is not None:
            iou50_idx = int(np.argmin(np.abs(coco_eval.params.iouThrs - 0.5)))
            for k, cid in enumerate(coco_eval.params.catIds):
                p50 = precision_array[iou50_idx, :, k, 0, 2]
                pall = precision_array[:, :, k, 0, 2]
                ap50_by_cat[cid] = float(np.mean(p50[p50 > -1])) if np.any(p50 > -1) else np.nan
                ap5095_by_cat[cid] = float(np.mean(pall[pall > -1])) if np.any(pall > -1) else np.nan

    total_tp = total_fp = total_fn = 0
    all_ious = []
    rows = []

    for cid in cat_ids:
        tp = fp = fn = 0
        matched_ious = []
        gt_count = 0

        for img_id in img_ids:
            gt_boxes = gt_by_img_cat.get((img_id, cid), [])
            preds = sorted(pred_by_img_cat.get((img_id, cid), []), key=lambda d: d["score"], reverse=True)
            pred_boxes = [xywh_to_xyxy(d["bbox"]) for d in preds]
            gt_count += len(gt_boxes)

            if pred_boxes and gt_boxes:
                ious = box_iou_xyxy(pred_boxes, gt_boxes)
                used_gt = np.zeros(len(gt_boxes), dtype=bool)
                for p_idx in range(len(pred_boxes)):
                    best_gt = int(np.argmax(ious[p_idx]))
                    best_iou = float(ious[p_idx, best_gt])
                    if best_iou >= iou_threshold and not used_gt[best_gt]:
                        used_gt[best_gt] = True
                        tp += 1
                        matched_ious.append(best_iou)
                    else:
                        fp += 1
                fn += int((~used_gt).sum())
            else:
                fp += len(pred_boxes)
                fn += len(gt_boxes)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0

        rows.append({
            "Class": cats[cid],
            "GT Count": gt_count,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "IoU": mean_iou,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "AP@0.50": ap50_by_cat.get(cid, np.nan),
            "AP@0.50:0.95": ap5095_by_cat.get(cid, np.nan),
        })
        total_tp += tp
        total_fp += fp
        total_fn += fn
        all_ious.extend(matched_ious)

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    mean_iou = float(np.mean(all_ious)) if all_ious else 0.0
    return {
        "IoU": mean_iou,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
    }, pd.DataFrame(rows)


def evaluate_detector(model, data_loader, coco_gt, split_name="valid"):
    detections, infer_time, image_count = collect_predictions(model, data_loader, score_min=0.001)
    if detections:
        coco_dt = coco_gt.loadRes(detections)
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.params.imgIds = sorted(coco_gt.getImgIds())
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
    else:
        coco_eval = None
        print("No detections; COCO AP metrics set to 0.")

    point_metrics, per_class_df = compute_point_metrics(
        coco_gt,
        detections,
        coco_eval,
        conf_threshold=CONF_THRESHOLD,
        iou_threshold=IOU_THRESHOLD,
    )
    if coco_eval is not None:
        coco_map_5095 = float(coco_eval.stats[0])
        coco_ap50 = float(coco_eval.stats[1])
        coco_ap75 = float(coco_eval.stats[2])
        coco_ar100 = float(coco_eval.stats[8])
    else:
        coco_map_5095 = coco_ap50 = coco_ap75 = coco_ar100 = 0.0

    fps = image_count / infer_time if infer_time > 0 else 0.0
    summary = {
        "Model": MODEL_NAME,
        "Split": split_name,
        "IoU": point_metrics["IoU"],
        "Precision": point_metrics["Precision"],
        "Recall": point_metrics["Recall"],
        "F1-score": point_metrics["F1-score"],
        "Average Precision (AP)": coco_ap50,
        "mean Average Precision (mAP)": coco_map_5095,
        "AP@0.75": coco_ap75,
        "AR@100": coco_ar100,
        "FPS": fps,
        "Conf threshold": CONF_THRESHOLD,
        "IoU threshold": IOU_THRESHOLD,
        "Classes": NUM_CLASSES,
        "Images": image_count,
    }
    return summary, per_class_df, detections


In [7]:
def make_optimizer(model):
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("backbone.body"):
            backbone_params.append(param)
        else:
            head_params.append(param)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": LR_BACKBONE})
    if head_params:
        groups.append({"params": head_params, "lr": LR_HEAD})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


optimizer = make_optimizer(model)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

last_ckpt_path = OUTPUT_DIR / "checkpoint.pth"
best_ckpt_path = OUTPUT_DIR / "best_checkpoint.pth"
history_path = OUTPUT_DIR / "history.json"
history = {"train_loss": [], "val_mAP": [], "val_AP50": []}
best_map = -1.0
start_epoch = 1

if RESUME_FROM_LAST and last_ckpt_path.exists():
    ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    lr_scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler is not None and ckpt.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_map = ckpt.get("best_map", best_map)
    history = ckpt.get("history", history)
    print(f"Resumed from epoch {start_epoch}; best validation mAP={best_map:.4f}")


def train_one_epoch(epoch):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for images, targets in pbar:
        images = [img.to(DEVICE) for img in images]
        targets = move_targets_to_device(targets, DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        if scaler is not None:
            scaler.scale(losses).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        loss_value = float(losses.detach().cpu())
        total_loss += loss_value
        pbar.set_postfix(loss=f"{loss_value:.4f}")

    return total_loss / max(1, len(train_loader))


coco_valid = valid_dataset.coco

if RUN_TRAIN:
    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(epoch)
        lr_scheduler.step()

        val_summary, _, _ = evaluate_detector(model, valid_loader, coco_valid, split_name="valid")
        val_map = val_summary["mean Average Precision (mAP)"]
        val_ap50 = val_summary["Average Precision (AP)"]
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_mAP"].append(val_map)
        history["val_AP50"].append(val_ap50)

        print(
            f"Epoch {epoch}/{EPOCHS} | {elapsed:.1f}s | "
            f"train_loss={train_loss:.4f} | val_mAP={val_map:.4f} | val_AP50={val_ap50:.4f}"
        )

        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": lr_scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "best_map": best_map,
            "history": history,
            "class_names": CLASS_NAMES,
            "model_name": MODEL_NAME,
            "config": {
                "image_min_size": IMAGE_MIN_SIZE,
                "image_max_size": IMAGE_MAX_SIZE,
                "freeze_backbone": FREEZE_BACKBONE,
                "unfreeze_last_stage": UNFREEZE_LAST_STAGE,
            },
        }
        torch.save(ckpt, last_ckpt_path)
        save_json(history, history_path)

        if val_map > best_map:
            best_map = val_map
            ckpt["best_map"] = best_map
            torch.save(ckpt, best_ckpt_path)
            print(f"Saved new best checkpoint: {best_ckpt_path}")
else:
    print("RUN_TRAIN=False; skipping training.")


Epoch 1/10:   0%|          | 0/3662 [00:00<?, ?it/s]

Inference:   0%|          | 0/1049 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a9ad11aa7a0>
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a9ad11aa7a0>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()
    self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

     if w.is_alive(): 
        ^  ^ ^ ^^^^^^^^^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'

   File "/usr/lib/pytho

Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=33.21s).
Accumulating evaluation results...
DONE (t=0.84s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.196
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.394
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.169
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.011
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.135
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.291
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.142
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.270
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.320
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

Epoch 2/10:   0%|          | 0/3662 [00:00<?, ?it/s]

...

Loading and preparing results...
DONE (t=10.64s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=21.41s).
Accumulating evaluation results...
DONE (t=0.56s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.318
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.531
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.321
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.056
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.240
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.443
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.170
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.340
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.406
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

Epoch 10/10:   0%|          | 0/3662 [00:00<?, ?it/s]

Inference:   0%|          | 0/1049 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a9ad11aa7a0><function _MultiProcessingDataLoaderIter.__del__ at 0x7a9ad11aa7a0>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()    self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive():Exception ignored in: Exception ignored in: 
if w.is_alive():<function _MultiProcessingDataLoaderIter.__del__ at 0x7a9ad11aa7a0><function _MultiProcessingDataLoaderIter.__del__ at 0x7a9ad11aa7a0>
 
 
 Traceback (most recent call last):


Loading and preparing results...
DONE (t=10.58s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=21.11s).
Accumulating evaluation results...
DONE (t=0.57s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.319
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.532
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.323
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.050
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.240
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.445
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.173
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.341
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.406
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

In [8]:
# Load best checkpoint and benchmark on the official test split.
checkpoint_path = best_ckpt_path if best_ckpt_path.exists() else last_ckpt_path
if not checkpoint_path.exists():
    raise FileNotFoundError(f"No checkpoint found at {best_ckpt_path} or {last_ckpt_path}. Run training first.")

ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()

coco_test = test_dataset.coco
test_summary, test_per_class_df, test_predictions = evaluate_detector(model, test_loader, coco_test, split_name="test")
test_summary["Checkpoint"] = str(checkpoint_path)

summary_df = pd.DataFrame([test_summary])
summary_path = BENCHMARK_DIR / f"{MODEL_SLUG}_test_summary.csv"
per_class_path = BENCHMARK_DIR / f"{MODEL_SLUG}_test_per_class.csv"
pred_path = BENCHMARK_DIR / f"{MODEL_SLUG}_test_predictions.json"
json_path = BENCHMARK_DIR / f"{MODEL_SLUG}_test_summary.json"

summary_df.to_csv(summary_path, index=False)
test_per_class_df.to_csv(per_class_path, index=False)
save_json(test_predictions, pred_path)
save_json(test_summary, json_path)

display(summary_df)
display(test_per_class_df)
print("Saved:")
print(" -", summary_path)
print(" -", per_class_path)
print(" -", pred_path)
print(" -", json_path)
print(" -", checkpoint_path)


Inference:   0%|          | 0/521 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=13.07s).
Accumulating evaluation results...
DONE (t=0.18s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.327
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.522
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.333
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.067
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.149
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.459
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.226
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.380
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.438
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

,Model,Split,IoU,Precision,Recall,F1-score,Average Precision (AP),mean Average Precision (mAP),AP@0.75,AR@100,FPS,Conf threshold,IoU threshold,Classes,Images,Checkpoint
0,DINOv3 ConvNeXt-Tiny + Faster R-CNN,test,0.811226,0.57719,0.654969,0.613624,0.522331,0.326949,0.332843,0.43818,24.439958,0.5,0.5,6,1042,/kaggle/working/outputs_dinov3_fasterrcnn/best...


,Class,GT Count,TP,FP,FN,IoU,Precision,Recall,F1-score,AP@0.50,AP@0.50:0.95
0,BIODEGRADABLE,49,15,100,34,0.708899,0.130435,0.306122,0.182927,0.126740,0.057518
1,CARDBOARD,20,12,65,8,0.761675,0.155844,0.600000,0.247423,0.458397,0.265940
2,GLASS,0,0,115,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
3,METAL,533,395,200,138,0.845416,0.663866,0.741088,0.700355,0.726125,0.509684
4,PAPER,1375,854,555,521,0.831928,0.606104,0.621091,0.613506,0.635341,0.423145
5,PLASTIC,1585,1057,674,528,0.783738,0.610630,0.666877,0.637515,0.665053,0.378456


Saved:
 - /kaggle/working/benchmark_results/dinov3_convnext_tiny_fasterrcnn_test_summary.csv
 - /kaggle/working/benchmark_results/dinov3_convnext_tiny_fasterrcnn_test_per_class.csv
 - /kaggle/working/benchmark_results/dinov3_convnext_tiny_fasterrcnn_test_predictions.json
 - /kaggle/working/benchmark_results/dinov3_convnext_tiny_fasterrcnn_test_summary.json
 - /kaggle/working/outputs_dinov3_fasterrcnn/best_checkpoint.pth
